# Prepare NDVI time-series data for OCC experiments
Create reproducible spatial splits and model-ready arrays. This notebook does not select or train an autoencoder.

In [ ]:
from pathlib import Path
import numpy as np
from greenwave_ndvi import open_ndvi_stack

# With years=None, the package discovers all raw observations in the local cache.
annual = open_ndvi_stack(years=None, chunks=(1, 512, 512))
annual

In [ ]:
# Require a valid value in every selected year for the first baseline.
complete = annual.valid.all('time')
series = annual.ndvi.where(complete).stack(pixel=('y', 'x')).dropna('pixel').transpose('pixel', 'time')
series

In [ ]:
# Split by projected x coordinate to reduce spatial leakage between sets.
x_coordinate = series.x
q_train, q_validation = np.quantile(x_coordinate, [0.60, 0.80])
train = series.where(x_coordinate <= q_train, drop=True)
validation = series.where((x_coordinate > q_train) & (x_coordinate <= q_validation), drop=True)
test = series.where(x_coordinate > q_validation, drop=True)
{name: value.sizes['pixel'] for name, value in {'train': train, 'validation': validation, 'test': test}.items()}

In [ ]:
output = Path('../../.cache/playground')
output.mkdir(parents=True, exist_ok=True)
np.savez_compressed(output / 'annual-ndvi-occ-splits.npz', train=train.compute().values, validation=validation.compute().values, test=test.compute().values, dates=annual.time.values)
